# Instruções

**A entrada é um csv, exemplo de estrutura abaixo:**

Nome;Conta;Cartao

Jone;0001;xxxx xxxx xxxx 0001

Anders;0002;xxxx xxxx xxxx 0002

A transformação usa open ia, através de gihub, devido às questões de crédito

O Carregamento simula disponibilizar em arquivo json, uma vez que não temos a api "sdw-2023-prd.up.railway.app"



# Extract

In [46]:
import pandas as pd
users = pd.read_csv('sd_teste.csv', sep=';').to_dict(orient='records')

# Garante a estrutura esperada para a etapa de Transformação
for user in users:
    user['news'] = []
print(users)

[{'Nome': 'Jone', 'Conta': 1, 'Cartao': 'xxxx xxxx xxxx 0001', 'news': []}, {'Nome': 'Anders', 'Conta': 2, 'Cartao': 'xxxx xxxx xxxx 0002', 'news': []}]


# Transform

In [35]:
!pip install openai

In [27]:
!pip install azure-ai-inference

In [29]:
# Documentação Oficial da API OpenAI: https://platform.openai.com/docs/api-reference/introduction
# Informações sobre o Período Gratuito: https://help.openai.com/en/articles/4936830

# Para gerar uma API Key:
# 1. Crie uma conta na OpenAI
# 2. Acesse a seção "API Keys"
# 3. Clique em "Create API Key"
# Link direto: https://platform.openai.com/account/api-keys

# Substitua o texto TODO por sua API Key da OpenAI, ela será salva como uma variável de ambiente.
openai_gitmodel_key = 'seu_tolken_git_aqui'

In [49]:
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential


def generate_ai_news(user):
    endpoint = "https://models.github.ai/inference"
    model = "openai/gpt-4o-mini"
    token = openai_gitmodel_key

    client = ChatCompletionsClient(
        endpoint=endpoint,
        credential=AzureKeyCredential(token),
    )

    response = client.complete(
        messages=[
          {
              "role": "system",
              "content": "Você é um especialista em educação digital."
          },
          {
              "role": "user",
              "content": f"Crie uma mensagem para {user['Nome']} sobre a importância algorítmos para estudantes de tecnologia (máximo 120 caracteres)"
          }
        ],
        model=model
    )
    return response.choices[0].message.content.strip('\"')

for user in users:
  news = generate_ai_news(user)
  print(news)
  user['news'].append({
      "description": news
  })
  print(user)
print(type(user))

Jone, compreender algoritmos é crucial para estudantes de tecnologia; eles são a base para resolver problemas e inovar!
{'Nome': 'Jone', 'Conta': 1, 'Cartao': 'xxxx xxxx xxxx 0001', 'news': [{'description': 'Oi Jone! Algoritmos são essenciais para estudantes de tecnologia, pois promovem raciocínio lógico e solução de problemas.'}, {'description': 'Jone, compreender algoritmos é crucial para estudantes de tecnologia; eles são a base para resolver problemas e inovar!'}]}
Anders, entender algoritmos é essencial para estudantes de tecnologia; eles são a base da lógica e resolução de problemas!
{'Nome': 'Anders', 'Conta': 2, 'Cartao': 'xxxx xxxx xxxx 0002', 'news': [{'description': 'Anders, entender algoritmos é crucial para estudantes de tecnologia; eles são a base da programação e resolução de problemas!'}, {'description': 'Anders, entender algoritmos é essencial para estudantes de tecnologia; eles são a base da lógica e resolução de problemas!'}]}
<class 'dict'>


# Load

In [58]:
import json

# Se 'users' já é sua lista de dicionários, salve direto:
with open('dados_atualizados.json', 'w', encoding='utf-8') as f:
    json.dump(users, f, ensure_ascii=False, indent=4)

print("Arquivo gerado diretamente da variável 'users'.")

Arquivo gerado diretamente da variável 'users'.
